In [19]:
import os
import pickle
import joblib

import streamlit as st
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import r2_score
import xgboost as xgb
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline,FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import(
OneHotEncoder,
    MinMaxScaler,
    PowerTransformer,
    FunctionTransformer,
    StandardScaler,
    OrdinalEncoder
)
from sklearn.ensemble import RandomForestRegressor
from feature_engine.selection import SelectBySingleFeaturePerformance
from feature_engine.datetime import DatetimeFeatures
from feature_engine.outliers import Winsorizer
from feature_engine.encoding import (RareLabelEncoder
                                    ,MeanEncoder,CountFrequencyEncoder)

import warnings

In [3]:
path=r"C:\Users\koush\anaconda3\flights-sagemaker\data\train.csv"
train=pd.read_csv(path)
X_train=train.drop(columns='price')
y_train=train.price.copy()

In [15]:
pd.set_option('display.max_columns',None)
sklearn.set_config(transform_output='pandas')
warnings.filterwarnings('ignore')

In [16]:
#airline
air_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy='most_frequent')),
    ("grouper",RareLabelEncoder(tol=0.1,replace_with='Other',n_categories=2)),
    ("encoder",OneHotEncoder(sparse_output=False,handle_unknown='ignore'))
])

#doj
feature_to_extract=["month","year","week","day_of_week","day_of_month","day_of_year"]

doj_transformer=Pipeline(steps=[
    ("dt",DatetimeFeatures(features_to_extract=feature_to_extract,yearfirst=True,format='mixed')),
    ('scaler',MinMaxScaler())
])

#source and destination
location_pipe1=Pipeline(steps=[
    ('grouper',RareLabelEncoder(tol=0.1,replace_with='Other',n_categories=2)),
    ('encoder',MeanEncoder()),
    ('scaler',PowerTransformer())   
])
def is_north(x):
    columns=x.columns.to_list()
    north_cities=['Delhi','Kolkata','New Delhi','Mumbai']
    return (
    x
    .assign(**{
        f"{col}_is_north": x.loc[:,col].isin(north_cities).astype(int)
        for col in columns
    })
    .drop(columns=columns))
location_transformer=FeatureUnion(transformer_list=[
    ("part1",location_pipe1),
    ("part2",FunctionTransformer(func=is_north))
])

# dep_time and arrival_time
time_pipe1=Pipeline(steps=[
    ('dt',DatetimeFeatures(features_to_extract=['hour','minute'])),
    ('scaler',MinMaxScaler())
])
def part_of_day(x,morning=4,noon=12,eve=16,night=20):
    columns=x.columns.to_list()
    x_temp=x.assign(**{
        col: pd.to_datetime(x.loc[:,col]).dt.hour
        for col in columns
    })
    
    return(
    x_temp.assign(**{
        f"{col}_part_of_day":np.select(
        [x_temp.loc[:,col].between(morning,noon,inclusive='left'),
        x_temp.loc[:,col].between(noon,eve,inclusive='left'),
        x_temp.loc[:,col].between(eve,night,inclusive='left')],
        ['morning','afternoon','evening'],default='night')
        
        for col in columns
        
    })
        .drop(columns=columns)
    )
time_pipe2=Pipeline(steps=[
    ('part',FunctionTransformer(func=part_of_day)),
    ('count',CountFrequencyEncoder()),
    ('scaler',MinMaxScaler())
])
time_transformer=FeatureUnion(transformer_list=[
    ("part1",time_pipe1),
    ("part2",time_pipe2)
])

#duration
def duration_of_day(x,short=200,medium=550,long=850):
    columns=x.columns.to_list()
    x_temp=x.assign(**{
        col: pd.to_datetime(x.loc[:,col]).astype('int64') # Convert nanoseconds to microseconds
        for col in columns
    })
    
    return(
    x_temp.assign(**{
        f"{col}_duration_of_day":np.select(
        [x_temp.loc[:,col].between(short,medium,inclusive='left'),
        x_temp.loc[:,col].between(medium,long,inclusive='left')],
        ['short','medium'],default='long')
        
        for col in columns
        
    })
        .drop(columns=columns)
    )
def is_over(X,value=1000):
    return (
    X
    .assign(**{
        f'duration_over_{value}': X.duration.ge(value).astype(int)
    })
    .drop(columns='duration'))

time_transformer=FeatureUnion(transformer_list=[
    ("part1",time_pipe1),
    ("part2",time_pipe2)
])
class RBFPercentileSimilarity(BaseEstimator,TransformerMixin):
    def __init__(self,variables=None,percentiles=[0.25,0.5,0.75],gamma=0.01):
        self.variables=variables
        self.percentiles=percentiles
        self.gamma=gamma
        
        
    def fit(self,X,y=None):
        if not self.variables:
            self.variables=X.select_dtypes(include='number').columns.to_list()
            
        self.reference_values={
            col: (
            X
                .loc[:,col]
                .quantile(self.percentiles)
                .values
                .reshape(-1,1)
            )
            for col in self.variables
        }
        
        return self
    
    def transform(self,X):
        objects=[]
        for col in self.variables:
            columns=[f"{col}_rbf_{int(percentile*100)}"for percentile in self.percentiles]
            obj=pd.DataFrame(
            data=rbf_kernel(X.loc[:,[col]],Y=self.reference_values[col],gamma=self.gamma),columns=columns)
            objects.append(obj)
            return pd.concat(objects,axis=1)
duration_pipe1=Pipeline(steps=[
    ('rbf',RBFPercentileSimilarity()),
    ('scaler',PowerTransformer())
])

duration_pipe2=Pipeline(steps=[
    ('cat',FunctionTransformer(func=duration_of_day)),
    ('encoder',OrdinalEncoder(categories=[['short','medium','long']]))
])
duration_union=FeatureUnion(transformer_list=[
    ('part1',duration_pipe1),
    ('part2',duration_pipe2),
    ('part3',FunctionTransformer(func=is_over)),
    ('part4', StandardScaler())
])
duration_transformer=Pipeline(steps=[
    ('outliers',Winsorizer(capping_method='iqr',fold=1.5)),
    ('imputer',SimpleImputer(strategy='median')),
    ('union',duration_union)
])

#total_stops
def is_direct(X):
    return X.assign(is_direct_flight=X.total_stops.eq(0).astype(int))


total_stops_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("", FunctionTransformer(func=is_direct))
])

#additional_info
info_pipe1 = Pipeline(steps=[
    ("group", RareLabelEncoder(tol=0.1, n_categories=2, replace_with="Other")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
def have_info(X):
    return X.assign(additional_info=X.additional_info.ne("No Info").astype(int))
info_union = FeatureUnion(transformer_list=[
    ("part1", info_pipe1),
    ("part2", FunctionTransformer(func=have_info))
])
info_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("union", info_union)
])

# column transformer
column_transformer=ColumnTransformer(transformers=[
    ("air",air_transformer,['airline']),
    ("doj",doj_transformer,['date_of_journey']),
    ("loc",location_transformer,['source','destination']),
    ("time",time_transformer,['dep_time','arrival_time']),
    ("duration",duration_transformer,['duration']),
    ("total_stops",total_stops_transformer,['total_stops']),
    ("info", info_transformer, ["additional_info"])
])

#feature selection
estimator = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)

selector = SelectBySingleFeaturePerformance(
    estimator=estimator,
    scoring="r2",
    threshold=0.1 #threshold for r2 score
)
preprocessor = Pipeline(steps=[
    ("ct", column_transformer),
    ("selector", selector)
])


In [18]:
preprocessor.fit(X_train,y_train)
joblib.dump(preprocessor,'preprocessor.joblib')

['preprocessor.joblib']

In [55]:
#web application
st.set_page_config(
page_title='Flights Price Prediction',
page_icon='✈️',
layout='wide'
)
st.title("Flights Prices Prediction - AWS SageMaker")

# user inputs
airline = st.selectbox(
	"Airline:",
	options=X_train.airline.unique()
)

doj = st.date_input("Date of Journey:")

source = st.selectbox(
	"Source",
	options=X_train.source.unique()
)

destination = st.selectbox(
	"Destination",
	options=X_train.destination.unique()
)

dep_time = st.time_input("Departure Time:")

arrival_time = st.time_input("Arrival Time:")

duration = st.number_input(
	"Duration (mins):",
	step=1
)

total_stops = st.number_input(
	"Total Stops:",
	step=1,
	min_value=0
)

additional_info = st.selectbox(
	"Additional Info:",
	options=X_train.additional_info.unique()
)

x_new = pd.DataFrame(dict(
	airline=[airline],
	date_of_journey=[doj],
	source=[source],
	destination=[destination],
	dep_time=[dep_time],
	arrival_time=[arrival_time],
	duration=[duration],
	total_stops=[total_stops],
	additional_info=[additional_info]
)).astype({
	col: "str"
	for col in ["date_of_journey", "dep_time", "arrival_time"]
})

#if st.button("Predict"):
saved_preprocessor = joblib.load("preprocessor.joblib")
x_new_pre = saved_preprocessor.transform(x_new)

path=r"C:\Users\koush\anaconda3\flights-sagemaker\xgboost-model (1)"
with open(path, "rb") as f:
	model = pickle.load(f)
x_new_xgb = xgb.DMatrix(x_new_pre)
pred = model.predict(x_new_xgb)

#st.info(f"The predicted price is {pred:,.0f} INR")

In [56]:
model.feature_names

In [77]:
saved_preprocessor = joblib.load("preprocessor.joblib")
x_new_pre = saved_preprocessor.transform(x_new)

In [79]:
x_new_pre.values

array([[ 0.        ,  1.        ,  0.        ,  0.00847458, -0.45567029,
        -0.44916845, -1.30792586,  0.        ,  1.        ]])

In [85]:
path=r"C:\Users\koush\anaconda3\flights-sagemaker\xgboost-model (1)"
with open(path, "rb") as f:
	model = pickle.load(f)
x_new_xgb = xgb.DMatrix(x_new_pre.values)
pred = model.predict(x_new_xgb)

In [94]:
model.num_features()

13

In [73]:
x_new_xgb = xgb.DMatrix(x_new_pre)
print(x_new_xgb.num_row(), x_new_xgb.num_col())

1 9


In [74]:
X_train_pre = preprocessor.transform(X_train)
print("Training shape:", X_train_pre.shape)

Training shape: (640, 9)


In [64]:
print("XGBoost Model Features:", model.feature_names)
print("X_new_pre Shape:", x_new_pre.shape)
print("X_new_xgb Columns:", x_new_xgb.num_col())

XGBoost Model Features: None
X_new_pre Shape: (1, 9)
X_new_xgb Columns: 9


In [88]:
print("X_new_xgb feature names:", x_new_xgb.feature_names)
print("Model feature names:", model.feature_names)

X_new_xgb feature names: None
Model feature names: None


In [66]:
X_train

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info
0,Jet Airways,2019-04-01,Kolkata,Banglore,06:30:00,16:20:00,590,1.0,No Info
1,Indigo,2019-06-06,Delhi,Cochin,14:20:00,22:30:00,490,1.0,No Info
2,Indigo,2019-05-06,Kolkata,Banglore,15:15:00,17:45:00,150,0.0,No Info
3,Air India,2019-03-01,Chennai,Kolkata,11:40:00,13:55:00,135,0.0,No Info
4,Indigo,2019-05-21,Banglore,Delhi,21:15:00,00:15:00,180,0.0,No Info
...,...,...,...,...,...,...,...,...,...
635,Jet Airways,2019-05-06,Kolkata,Banglore,14:05:00,09:20:00,1155,1.0,In-flight meal not included
636,Jet Airways,2019-05-06,Kolkata,Banglore,17:00:00,23:35:00,395,1.0,No Info
637,Jet Airways,2019-05-12,Kolkata,Banglore,16:30:00,22:35:00,365,1.0,In-flight meal not included
638,Vistara,2019-03-18,Banglore,New Delhi,16:00:00,18:35:00,155,0.0,No Info


In [68]:
print(preprocessor)

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('air',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('grouper',
                                                                   RareLabelEncoder(n_categories=2,
                                                                                    replace_with='Other',
                                                                                    tol=0.1)),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['airline']),


In [69]:
print("Shape of preprocessed training data:", preprocessor.transform(X_train).shape)
print("Shape of new input data:", x_new_pre.shape)

x_new_xgb = xgb.DMatrix(x_new_pre)
print("Shape of x_new_xgb:", x_new_xgb.num_col())

Shape of preprocessed training data: (640, 9)
Shape of new input data: (1, 9)
Shape of x_new_xgb: 9


In [70]:
print("Training DMatrix feature names:", dtrain.feature_names)

NameError: name 'dtrain' is not defined

In [71]:
print("Model feature names:", model.feature_names)
print("Input feature names:", list(x_new_pre.columns))

Model feature names: None
Input feature names: ['air__airline_Indigo', 'air__airline_Jet Airways', 'doj__date_of_journey_week', 'doj__date_of_journey_day_of_year', 'loc__destination', 'duration__duration_rbf_25', 'duration__duration', 'total_stops__total_stops', 'total_stops__is_direct_flight']
